In [1]:
!pip -q install transformers accelerate

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
set_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
# ---------- Chatbot 1: DialoGPT (conversational GPT-2) ----------
tok = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium").to(device).eval()

class DialoBot:
    def __init__(self):
        self.history = None

    def reply(self, text):
        new_ids = tok.encode(text + tok.eos_token, return_tensors="pt").to(device)
        inp = new_ids if self.history is None else torch.cat([self.history, new_ids], dim=-1)
        with torch.no_grad():
            self.history = model.generate(inp, attention_mask=torch.ones_like(inp), max_length=1000,
                                          pad_token_id=tok.eos_token_id, do_sample=True,
                                          top_k=50, top_p=0.9, temperature=0.7)
        return tok.decode(self.history[0, inp.shape[-1]:], skip_special_tokens=True)

bot = DialoBot()
for msg in ["Hello, how are you?", "What do you like to do in your free time?",
            "I am a student and I love programming.", "Can you give me a tip for exams?"]:
    print("You:", msg)
    print("Bot:", bot.reply(msg), "\n")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  863MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

You: Hello, how are you?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Bot: Fine . I guess . 

You: What do you like to do in your free time?
Bot: I do nothing . 

You: I am a student and I love programming.
Bot: Cool . Well , I'm a student and I love programming . 

You: Can you give me a tip for exams?
Bot: I'm not good at that . 



In [4]:
# ---------- Interactive chat (type 'quit' to stop) ----------
bot = DialoBot()
while True:
    user = input("You: ")
    if user.strip().lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye!")
        break
    print("Bot:", bot.reply(user))

You: how are you
Bot: I'm good .
You: what is your name
Bot: Kris . She says .
You: ok
Bot: She nods .
You: vyee
Bot: She nods back .
You: bye
Bot: Goodbye!


In [5]:
# ---------- Chatbot 2: instruction-tuned small LLM (ChatGPT-style) ----------
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok2 = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto").to(device).eval()

messages = [{"role": "system", "content": "You are a helpful assistant for college students. Keep answers short."}]

def chat(user_text, max_new_tokens=120):
    messages.append({"role": "user", "content": user_text})
    prompt = tok2.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tok2(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    answer = tok2.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True)
    messages.append({"role": "assistant", "content": answer})
    return answer

for q in ["What is a transformer in NLP?", "Explain it in one simple example.", "Suggest 3 NLP project ideas."]:
    print("You:", q)
    print("Bot:", chat(q), "\n")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

You: What is a transformer in NLP?
Bot: A transformer is an advanced neural network model used in Natural Language Processing (NLP) to understand and generate human language. It consists of multiple layers of processing, including the encoder-decoder architecture, which allows it to capture long-term dependencies between words. Transformers excel at tasks like question answering, text summarization, and machine translation by leveraging their ability to efficiently process large amounts of data quickly. 

You: Explain it in one simple example.
Bot: Imagine you have a big box of crayons that can be rearranged into different shapes using your imagination. A transformer works similarly but with words instead of crayons. When you give it a sentence or piece of text, it uses its magical brain to break down the meaning and structure of those words, just like how a real brain does when you tell it about a story. This makes it super good at understanding and generating complex sentences, even 

In [ ]:
# ---------- Interactive chat with the instruction model ----------
while True:
    user = input("You: ")
    if user.strip().lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye!")
        break
    print("Bot:", chat(user))

You: hey
Bot: Hello! How may I assist you today?
You: what is today weather
Bot: I'm sorry, but I don't actually see anything labeled "today" on the page you're referring to. Could you please describe what you mean by "weather"?
You: what you can do
Bot: As an AI language model, I am designed to provide information and answer questions to the best of my abilities within the parameters set forth by my creators. However, I cannot physically observe or access external conditions outside of my training data. My responses are based solely on the knowledge contained within my programming and the input provided to me. Is there something specific you would like assistance with?
You: byee
Bot: Bye! If you need any further assistance, feel free to ask.
You: ok
Bot: Goodbye! If you have any other questions or need help with anything else, please don't hesitate to reach out.
